# SaaS Churn & Revenue Retention — Analysis

End-to-end business analytics build: churn quantification, revenue impact, engagement, and cohorts.

- **Dataset:** `Data/Churn Analysis Final.csv`
- **Methodology:** See `CALCULATIONS.md`

## Objectives
1. Compute overall churn rate
2. Quantify revenue impact (MRR / ARR)
3. Analyze engagement vs. churn
4. Cohort-based churn analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '${:,.0f}'.format)

DATA_PATH = "Data/Churn Analysis Final.csv"
df = pd.read_csv(DATA_PATH)
df["Churn_flag"] = df["Churn_flag"].astype(str).str.upper().eq("TRUE")
print(f"Shape: {df.shape}")

In [ ]:
# 1. Churn rate + revenue impact
churn_rate = df["Churn_flag"].mean()
churned = df[df["Churn_flag"]]
retained = df[~df["Churn_flag"]]

mrr_total = df["Mrr_amount"].sum()
arr_total = df["Arr_amount"].sum()
mrr_churned = churned["Mrr_amount"].sum()
arr_churned = churned["Arr_amount"].sum()

print(f"Churn Rate: {churn_rate:.2%}")
print(f"MRR Total: ${mrr_total:,.0f}")
print(f"ARR Total: ${arr_total:,.0f}")
print(f"Churned MRR: ${mrr_churned:,.0f}")
print(f"Churned ARR: ${arr_churned:,.0f}")

In [ ]:
# Churned vs retained revenue
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(x=["Retained", "Churned"], y=[retained["Mrr_amount"].sum(), churned["Mrr_amount"].sum()], ax=ax)
ax.set_title("MRR: Retained vs Churned")
ax.set_ylabel("MRR ($)")
plt.show()

In [ ]:
# Engagement: churn by usage quartile
df["usage_quartile"] = pd.qcut(df["Usage_count"], 4, labels=["Low", "Low-Mid", "Mid", "High"])
churn_by_usage = df.groupby("usage_quartile", observed=True)["Churn_flag"].mean()
churn_by_usage.plot(kind="bar", ax=plt.gca())
plt.title("Churn Rate by Usage Quartile")
plt.ylabel("Churn Rate")
plt.show()

In [ ]:
# Cohorts: churn by signup month
df["Signup_month"] = pd.to_datetime(df["Signup_date"], errors="coerce", dayfirst=True).dt.to_period("M").astype(str)
cohort = df.groupby("Signup_month").agg(
    accounts=("Churn_flag", "size"),
    churned=("Churn_flag", lambda x: x.sum()),
).reset_index()
cohort["churn_rate"] = cohort["churned"] / cohort["accounts"]
cohort.sort_values("churn_rate", ascending=False).plot(x="Signup_month", y="churn_rate", kind="bar", ax=plt.gca())
plt.title("Churn Rate by Signup Cohort")
plt.ylabel("Churn Rate")
plt.xticks(rotation=45)
plt.show()